# Lab 4

# JavaScript & Leaflet in Practice: Building and Publishing Your Own Web Map
### GISS/GEOG 366/368 · Web Mapping & Web GIS

**Unit 5 Focus:** JavaScript and Leaflet, from a blank file. Unlike Lab 3, there is no provided `map.js` today; you write it. This is your fourth graded lab, and the first time you've authored the map's actual behavior yourself.

## Before You Begin

The tools are the same as Lab 3. Confirm you still have them before proceeding.

| Tool | Role today |
|---|---|
| **A GitHub account** | Already confirmed in Lab 3 |
| **VS Code, with the Git/GitHub extension enabled** | Cloning the Lab 4 template and editing files locally |
| **Git** | Same install as Lab 3 |
| **VS Code + Live Server** | Previewing your map locally as you write JavaScript, before pushing |
| **The Lab 4 starter repository** | Provided by your instructor; a fresh template, similar in structure to the Lab 3 template but with an empty `map.js` |
| **A region and a few locations of your own** | You'll need real (or realistic) coordinates to place on your map (see Step 1) |
| **[MDN Web Docs](https://developer.mozilla.org/)** and **[Leaflet documentation](https://leafletjs.com/reference.html)** | Your first stops when something doesn't work |

You already know how to clone a repo, edit locally, preview with Live Server, and publish via GitHub Pages from Lab 3. That workflow doesn't change today. What's new is the file you're actually writing code into.

## Step 1: Ask

Open this reference map for a minute: https://leafletjs.com/examples/quick-start/

This is the map Dorman's Chapter 6 walks through building, and it's close to what you're building today, minus the styling and customization you'll add.

Before opening the starter repository, jot your first-pass answers:
- Pick a place that matters to your project topic, your hometown, a field site, a neighborhood you've mapped before in this course. What is its approximate latitude and longitude? (A quick search of "[place name] coordinates" will get you close enough.)
- List two or three additional nearby points, paths, or areas you might want to mark on your map, this is your rough content plan for Step 4.
- In Lab 3, you never opened `map.js`. Looking at the quick-start map above, take a guess: how many lines of JavaScript do you think it takes to get a tile layer and one marker on screen?

> **Lab 4 Question: In Lab 3, changing `#title`'s CSS rule changed the page without ever touching `map.js`. Today, `map.js` is the file that puts anything geographic on the page at all; no HTML or CSS on its own will draw a marker or a tile. Why do you think that division of labor exists between the three languages?**

## Step 2: Collect

### Part A: Confirm access

Same as Lab 3: confirm you can log in to GitHub, and that you have the link to the Lab 4 starter repository (shared via Canvas prior to lab).

- https://github.com/asivitskis/giss366-lab04-template

<details>
<summary><b>Part B: Make your own copy of the repository, then clone it in VS Code. Click to expand</b></summary>

This is the same process as Lab 3; if it's still fresh, skim and move on.

**Step 0: Create your own copy of the template repository.**

1. In a browser, go to the template repository URL above.
2. Make sure you're signed into your GitHub account.
3. Click **Use this template** → **Create a new repository**.
4. Name it something like `giss-366-lab4`, under **your own** account.
5. Click **Create repository**.
6. Copy the URL of **this new repository** (not the original template).

**Clone your own copy into VS Code:**

1. Open VS Code, no files open (File → New Window if needed).
2. Open the **Source Control** panel and click **Clone Repository**.
3. Paste your new repository's URL, choose a save location near your other `giss-366` folders.
4. Choose **Open** when prompted.

If you don't see **Use this template**, you're on the wrong page; check with Alex before proceeding.

</details>

### Part C: Explore what you cloned

| File / folder | What it is | Will you edit it today? |
|---|---|---|
| `index.html` | The full map page skeleton: `<head>`, Leaflet CSS/JS links, an empty `<div id="map">`, and a link to `map.js` | Minimal; mostly title text and `<style>` tweaks |
| `map.js` | **Empty.** This is where all of today's work happens | **Yes, this is the whole point** |
| `README.md` | Standard repository landing page | Optional |
| `css/leaflet.css` | Leaflet's own stylesheet | No |
| `js/leaflet.js` | The Leaflet library itself | No |
| `images/` | A folder for any marker icons or logo images you want to add | Optional |

The reversal from Lab 3 is the whole point of this lab: last time, two files were yours and two weren't, with `map.js` locked. Today `map.js` is the one file that matters most.

## Step 3: Visualize

### 3.1 The minimum a Leaflet map needs

*(Dorman, M. Introduction to Web Mapping, Sections 6.5.2–6.5.11)*

Open `index.html` in your cloned repository. It already includes the Leaflet library and an empty map `<div>`:

```html
<!DOCTYPE html>
<html>
<head>
    <title>[update this] My Map</title>
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
    <link rel="stylesheet" href="css/leaflet.css">
    <script src="js/leaflet.js"></script>
    <style>
        body { padding: 0; margin: 0; }
        html, body, #map { height: 100%; width: 100%; }
    </style>
</head>
<body>
    <div id="map"></div>
    <script src="map.js"></script>
</body>
</html>
```

Nothing will appear when you open this in a browser yet, the `<div id="map">` is empty, and `map.js`, the file that's supposed to fill it, doesn't exist yet. That's your job in Step 4.

### 3.2 Reading the shape of a Leaflet call

Every Leaflet layer you'll write today follows the same two-part shape: **create the layer, then add it to the map.**

```js
L.tileLayer(url, options).addTo(map);
L.marker([lat, lng]).addTo(map);
L.polyline([[lat, lng], [lat, lng]], options).addTo(map);
```

The `map` variable itself has to exist before any of these calls run, which means the very first line of your `map.js` is always going to be an `L.map(...)` call. Everything else in the file depends on that variable.

### 3.3 The full target file

Here's the complete pattern your `map.js` will follow by the end of Step 4 (values are placeholders, yours will use your own coordinates and styling):

```js
// 1. Create the map, centered on your chosen location
let map = L.map("map", {center: [45.5152, -122.6784], zoom: 12});

// 2. Add a basemap tile layer
L.tileLayer(
    "https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    {attribution: '&copy; OpenStreetMap contributors'}
).addTo(map);

// 3. Add at least one marker, line, or polygon
let pnt = L.marker([45.5152, -122.6784]).addTo(map);
pnt.bindPopup("Something worth saying about this location.");

// 4. Add a description control
let description = L.control({position: "bottomleft"});
description.onAdd = function() {
    let div = L.DomUtil.create("div", "description");
    div.innerHTML =
        "<p><b>[Your Map Title]</b></p><hr>" +
        "<p>A sentence or two about what this map shows.</p>";
    return div;
};
description.addTo(map);
```

Skim this before writing your own; you'll fill in each numbered section separately in Step 4.

## Step 4: Create

<details>
<summary><b>Part 1: Center your map and add a tile layer. Click to expand</b></summary>

Open `map.js` (currently empty) in VS Code.

1. Look up the approximate latitude and longitude for the place you chose in Step 1.
2. Write the first line of `map.js`, creating your map object centered there:
   ```js
   let map = L.map("map", {center: [YOUR_LAT, YOUR_LNG], zoom: 12});
   ```
3. Below it, add an `L.tileLayer` call to bring in an OpenStreetMap basemap (Lecture Part 7). Don't forget `.addTo(map)` at the end.
4. Save, then open `index.html` with Live Server. You should see a basemap, centered and zoomed where you specified, with nothing else on it yet.

**Try it:** Change only the `zoom` number and refresh. Then change only the `center` array. Confirm you understand which number controls which behavior before moving on.

</details>

<details>
<summary><b>Part 2: Add your own vector layers. Click to expand</b></summary>

Using the coordinates you listed in Step 1, add **at least two** of the following (mix and match; you don't need all three):

- **A marker** at one point of interest, with `.bindPopup()` containing at least one sentence and one HTML tag (`<b>`, `<a>`, etc.)
- **A line** connecting two or more of your points, with a custom `color` and `weight`
- **A polygon** outlining an area relevant to your topic, with a custom `color` and `fillColor`

For each one, follow the create-then-`.addTo(map)` pattern from Step 3.2. Check the map in Live Server after each addition, rather than writing all three and debugging at once.

**Common early bugs to check for, if nothing appears:**
- Did you swap latitude and longitude? Leaflet coordinate arrays are `[lat, lng]`.
- Is every statement inside `map.js` ending in a semicolon?
- Does every opening `{` or `(` have a matching closing `}` or `)`? (Check your browser's console: `F12` → Console tab, for a red error message pointing at the exact line.)

</details>

<details>
<summary><b>Part 3: Add a description control. Click to expand</b></summary>

Using the `L.control` pattern from Lecture Part 8, add a description panel to the bottom-left corner of your map with:

- A title (`<b>` tag) naming your map
- One or two sentences about what it shows and why
- A short list (`<ul>`/`<li>`) of the layers you added in Part 2

Then, in `index.html`'s `<style>` block, add a `.description` CSS rule to style the panel, reusing whatever you're comfortable with from Lab 3 (`background-color`, `padding`, `border-radius`, `font-family`, and so on all still apply here; `L.control` panels are styled with ordinary CSS, same as any other page element).

</details>

<details>
<summary><b>Part 4: Title and polish. Click to expand</b></summary>

Back in `index.html`:
- Update `<title>` to name your map.
- Confirm the `<meta name="viewport">` line is present (it should already be in the template); this keeps your map usable on mobile.

Preview the whole thing once more in Live Server before moving to Step 5. This is the version that gets pushed and graded.

</details>

<details>
<summary><b>Part 5: Publish to GitHub Pages. Click to expand</b></summary>

Same three-step process as Lab 3:

1. **Stage and commit**, using VS Code's Source Control panel. Stage `index.html` and `map.js`, write a commit message (e.g., `Build custom Leaflet map for Lab 4`), and commit.
2. **Push**, using Sync Changes / Push.
3. **Enable GitHub Pages**, in your repository's Settings → Pages (source: `main` branch, `/ (root)` folder), if it isn't already enabled from the template.
4. **Find your published URL** on the same Settings → Pages screen once the build finishes.

Open the published URL and confirm your map, markers, popups, and description panel all appear exactly as they did in Live Server.

</details>

## Step 5: Act

### Discussion 1: Access & Equity

With your own published map open in front of you, respond to the Discussion 1 prompt (posted separately on Canvas), grounded specifically in the map you just built:

- Who can actually open and use this map? Does it require a login, a specific device, or a fast connection?
- What did you rely on (a CDN, a specific tile provider, GitHub Pages itself) that could become unavailable or restricted, and what would that mean for someone trying to reach your map?
- Is there a licensing or attribution obligation built into anything you used today (tile provider, marker icons)? Did you meet it?

### What to turn in (graded: Lab Exercises)

1. Your completed `map.js`, pushed to your repository (Create, Parts 1–3).
2. Your updated `index.html`, pushed to your repository (Create, Parts 3–4).
3. The link to your published GitHub Pages URL (Create, Part 5).
4. The link to your GitHub repository (Collect, Part B).
5. Your Discussion 1 post (separate submission space on Canvas, per the discussion instructions).
6. A short written reflection (3–4 sentences) answering:
   - Revisit your Step 1 guess about line count. How close were you, and what surprised you about how little (or how much) code it took?
   - Name one specific bug you hit while writing `map.js` today, and how you found and fixed it (console error, mismatched bracket, swapped coordinates, etc.).
   - Compare today to Lab 3: which felt more like "real programming" to you, and why?

Submit your GitHub links and written reflection to this week's submission space in Canvas.

---

## Lab 4 Rubric (40 pts)

| Score Band | What It Looks Like |
|:---|:---|
| **Exceptional (7-8)** | Exceeding expectations; indicates mastery; in-depth understanding; higher-order thinking; inferences and extensions of learning objectives that go beyond what was taught; truly superb effort. |
| **Proficient (5-6)** | Meeting expectations; application of concepts; independently demonstrates understanding and thorough competency of learning objectives explicitly taught. |
| **Developing (3-4)** | Approaching expectations; demonstration of basic understanding without application and understanding of more complex ideas and processes; meets minimum requirements for satisfactory learning. |
| **Insufficient (1-2)** | Below expectations; partial or no demonstration of understanding and progress toward learning objectives; major errors and omissions present; inadequate for competency. |

<br>

| Criteria | Comments | Grade |
|:---|:---|:---:|
| **Content:** `map.js` correctly initializes a map, adds a tile layer, and adds at least two vector layers (marker/line/polygon) with a working popup. | | / 8 |
| **Content:** A custom `L.control` description panel is present, styled with CSS, and accurately describes the map's contents. | | / 8 |
| **Process:** The published GitHub Pages URL loads correctly and matches the local Live Server preview; the repository history shows a real commit/push, not just a single upload. | | / 8 |
| **Process:** The written reflection is specific and evidence-based, naming an actual bug encountered and how it was resolved. | | / 8 |
| **Product:** `map.js`, `index.html`, the GitHub Pages link, the repository link, and the reflection are all submitted, clearly labeled, and free of errors. | | / 8 |
| **Total** | | **/ 40** |

---

### Lab 4 AI Policy: Level 2

This lab permits optional use of AI to support student learning. Permitted uses can include using AI to help troubleshoot GIS workflows and debug JavaScript errors (for example, pasting a console error message to ask what it means), however, independent installation and use of AI agents to complete GIS tasks is not permitted at this time. AI may also be used to support general writing procedures (outlining a project, improving clarity or grammar, etc.), but all code and written responses must remain original to each student as a sole responsible author. Disclosure of AI use is not required for this Level 2 assignment. Using AI is a choice with ethical considerations, and it is asked that students weigh those considerations accordingly as they complete this work.

### Looking Ahead

Week 6 rebuilds this same map in MapLibre GL JS, framed as "same architecture, newer library," a direct transfer exercise against the map you just published, so you can see exactly what carries over and what's different in a newer mapping library.

---
### Resources

- [MDN Web Docs](https://developer.mozilla.org/)
- [Leaflet documentation](https://leafletjs.com/reference.html)
- [Leaflet Provider Demo](https://leaflet-extras.github.io/leaflet-providers/preview/) (alternative basemaps, if you'd like to swap the default OpenStreetMap tiles)
- [GitHub Docs: Configuring a publishing source for your GitHub Pages site](https://docs.github.com/en/pages/getting-started-with-github-pages/configuring-a-publishing-source-for-your-github-pages-site)
- Dorman, M. *[Introduction to Web Mapping](https://geobgu.xyz/web-mapping/)*, Chapter 3 (JavaScript Basics) and Chapter 4 (JavaScript Interactivity)
- Dorman, M. *[Introduction to Web Mapping](https://geobgu.xyz/web-mapping/)*, Chapter 6 (Leaflet), Sections 6.5–6.8